In [ ]:
# Al inicio del notebook, agregar estas líneas:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass, replace, asdict, field
from typing import get_type_hints
from dataclasses import fields
from typing import List, Dict
from functools import wraps
from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import pickle
import time
import csv

from RRAM import (
    CurrentSolver,
    Representate,
    exceptions,
    Percolation,
    Temperature,
    ElectricField,
    Generation,
    Recombination,
    utils
)

In [ ]:
def medir_tiempo(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        inicio = time.time()
        resultado = func(*args, **kwargs)
        fin = time.time()
        print(f"Función '{func.__name__}' ejecutada en {fin - inicio:.6f} segundos\n")
        return resultado
    return wrapper


In [ ]:
def read_csv_to_dic(cvs_path: str) -> List[Dict[str, str]]:
    """
    Reads a CSV file and returns its contents as a list of dictionaries.
    Args:
        cvs_path (str): The name of the CSV file to read.

    Returns:
        list: A list of dictionaries, where each dictionary represents a row in the CSV file.
    """
    with open(cvs_path, mode="r") as archivo:
        reader = csv.DictReader(archivo)
        data = [row for row in reader]
    return data


In [ ]:
@dataclass
class SimulationParameters:
    device_size: float
    atom_size: float
    x_size: int
    y_size: int
    num_trampas: int
    init_simulation_time: float
    total_simulation_time: float
    num_pasos: int
    paso_guardar: int
    voltaje_final_reset: float
    voltaje_final_set: float
    initial_voltaje: float
    initial_current: float
    initial_elec_field: float
    initial_temperatura: float
    T_0: float

    num_max_vacantes: int = field(init=False)
    paso_temporal: float = field(init=False)
    paso_potencial: float = field(init=False)

    def __post_init__(self):
        self.num_max_vacantes = int(0.9 * (self.device_size / self.atom_size) ** 2)
        self.paso_temporal = self.total_simulation_time / self.num_pasos
        self.paso_potencial = self.voltaje_final_reset / self.num_pasos

    def __repr__(self):
        # Crear lista de líneas con "nombre=valor" para cada atributo
        atributos = []
        # Usar vars(self) para incluir también campos calculados en __post_init__
        for nombre, valor in vars(self).items():
            atributos.append(f"   {nombre}={valor}")
        # Formatear en varias líneas
        return "Los parámetros de la simulación son:\n" + ",\n".join(atributos) + "\n"

    @staticmethod
    def from_dict(d: dict):
        # Usar get_type_hints para asegurar que obtienes tipos reales

        field_types = get_type_hints(SimulationParameters)
        init_fields = {f.name for f in fields(SimulationParameters) if f.init}

        mapping = {
            "voltaje_final_reset": "voltaje_final",
            "T_0": "init_temp",
            "initial_temperatura": "init_temp",
        }

        kwargs = {}
        for k in init_fields:
            src = mapping.get(k, k)

            # Verificar que la clave existe en el diccionario
            if src not in d:
                raise KeyError(f"La clave '{src}' no existe en el diccionario")

            # Debug: ver qué tipo y valor estamos procesando
            # print(f"Campo: {k}, Tipo: {field_types[k]}, Fuente: {src}, Valor: {d[src]}")

            try:
                kwargs[k] = field_types[k](d[src])
            except Exception as e:
                print(f"Error al convertir {k}: {e}")
                print(
                    f"Tipo esperado: {field_types[k]}, tipo real: {type(field_types[k])}"
                )
                raise

        return SimulationParameters(**kwargs)

In [ ]:
@dataclass(frozen=True)
class SimulationConstants:
    vibration_frequency: float
    migration_energy: float
    drift_coefficient: float
    cte_red: float
    recom_enchancement_factor: float
    decaimiento_concentracion: float
    activation_energy: float
    gamma: float
    ohm_resistence: float
    pb_metal_insul: float
    permitividad_relativa: float
    I_0: float
    r_termica_percola: float
    r_termica_no_percola: float
    factor_generacion: float
    recombination_energy: float
    num_oxigenos_pp_reset_1 : int
    num_oxigenos_pp_reset_2: int
    num_oxigenos_sp_reset: int

    @staticmethod
    def from_dict(d: dict):
        # Obtiene los tipos reales de SimulationConstants, no de SimulationParameters
        field_types = get_type_hints(SimulationConstants)

        # Alias si es necesario; si no, mapa vacío sirve
        mapping = {}

        kwargs = {}
        for k in field_types:
            src = mapping.get(k, k)

            if src not in d:
                raise KeyError(f"La clave '{src}' no existe en el diccionario")

            try:
                kwargs[k] = field_types[k](d[src])
            except Exception as e:
                print(f"Error al convertir {k} con valor {d[src]}: {e}")
                raise

        return SimulationConstants(**kwargs)
    
    def update_gamma(self, nuevo_valor_gamma: float):
        return replace(self, gamma=nuevo_valor_gamma)

    def __repr__(self):
        # Crear lista de líneas con "nombre=valor" para cada atributo
        atributos = []
        # Usar vars(self) para incluir también campos calculados en __post_init__
        for nombre, valor in vars(self).items():
            atributos.append(f"   {nombre}={valor}")
        # Formatear en varias líneas
        return "Las constantes de la simulación son:\n" + ",\n".join(atributos) + "\n"

In [ ]:
def crear_rutas_simulacion(num_simulation: int, state: str) -> dict:
    """
    Crea las rutas necesarias para guardar resultados de la simulación.

    Args:
        num_simulation (int): Número índice de la simulación.
        state (str): Estado de la simulación, puede ser 'set' o 'reset'.

    Returns:
        dict: Diccionario con las rutas Path para 'simulation', 'figures' y 'set'.

    Side-effects:
        Crea las carpetas en disco si no existen.
    """
    
    simulation_path = Path.cwd() / "Results copy" / f"simulation_{num_simulation}"
    figures_path = simulation_path / "Figures"
    data_simulation_path = simulation_path / state

    simulation_path.mkdir(parents=True, exist_ok=True)
    figures_path.mkdir(parents=True, exist_ok=True)
    data_simulation_path.mkdir(parents=True, exist_ok=True)

    return {
        "simulation_path": simulation_path,
        "figures_path": figures_path,
        "data_simulation_path": data_simulation_path,
    }

In [ ]:
def cargar_y_representar_estado(pkl_path: Path , figures_path: Path, voltage: float, plot_state: bool = True) -> np.ndarray:
    """
    Carga el estado de configuración desde archivo pkl y genera una imagen de ese estado.

    Args:
        pkl_path (Path): Ruta del pkl con el estado.
        figures_path (Path): Ruta donde se guardará la imagen generada, debe incluir el nombre del archivo CON extensión.
        plot_state (bool): Indica si se debe generar la imagen del estado, por defecto es True.

    Returns:
        np.ndarray: Matriz con el estado inicial cargado.
    """
    with open(f"{pkl_path}.pkl", "rb") as f:
        actual_state = pickle.load(f)
        
    if plot_state:
        Representate.RepresentateState(actual_state, voltage, str(figures_path))

    return actual_state

In [ ]:
def procesar_filamentos_creados(
    imagen_path,
    pkl_path,
    existentes,
    CF_creado,
    voltage,
    voltage_CF_creado,
    actual_state,
    num_simulation,
) :
    """
    Detecta filamentos nuevos, actualiza su estado y guarda imágenes e
    imágenes correspondientes, además guarda el estado actual en PKL.

    Args:
        filamentos (list): Lista de filamentos en el estado actual.
        CF_creado (np.ndarray): Vector booleano que indica si cada filamento fue creado.
        voltage (float): Voltaje actual.
        voltage_CF_creado (np.ndarray): Array para registrar voltajes de creación.
        actual_state (np.ndarray): Estado actual del sistema.
        num_simulation (int): Número de simulación.

    Returns:
        int: Índice actualizado para el filamento.
    """
    
    filamentos_nuevos = [i for i, v in enumerate(existentes) if v and not CF_creado[i]]
    
    for i in filamentos_nuevos:
        CF_creado[i] = True
        voltage_CF_creado[i] = voltage
        print(f"El filamento {i + 1} se ha creado en el voltaje {round(voltage,4)} (V)")

        nombre_img = (imagen_path / f"Filamento_{i + 1}_creado_set_{num_simulation}.png")
        
        Representate.RepresentateState(actual_state, round(voltage, 3), str(nombre_img))
        
        # Guardar estado actual en archivo pkl
        nombre_pkl = pkl_path / f"filamento_{i + 1}_creado_set_{num_simulation}.pkl"

        with open(nombre_pkl, "wb") as f:
            pickle.dump(actual_state, f)

    return None

In [ ]:
def procesar_filamentos_destruidos(
    imagen_path,
    pkl_path,
    existentes,
    CF_destruido,
    voltage,
    voltage_CF_destruido,
    actual_state,
    num_simulation,
):
    """
    Detecta filamentos rotos, actualiza su estado y guarda imágenes e
    imágenes correspondientes, además guarda el estado actual en PKL.

    Args:
        existentes (list[bool]): Lista booleana indicando existencia actual de filamentos.
        CF_destruido (np.ndarray): Vector booleano que indica si cada filamento fue destruido.
        voltage (float): Voltaje actual.
        voltage_CF_destruido (np.ndarray): Array para registrar voltajes de destrucción.
        actual_state (np.ndarray): Estado actual del sistema.
        num_simulation (int): Número de simulación.
        imagen_path (Path): Ruta donde guardar imágenes.
        pkl_path (Path): Ruta donde guardar archivos PKL.

    Returns:
        None
    """
    filamentos_rotos = [
        i for i, v in enumerate(existentes) if not v and not CF_destruido[i]
    ]

    for i in filamentos_rotos:
        CF_destruido[i] = True
        print(" El voltaje del filamento roto es:", voltage_CF_destruido[i])
        if voltage_CF_destruido[i] == 0:
            voltage_CF_destruido[i] = voltage
        print(f"El filamento {i + 1} se ha roto en el voltaje {round(voltage, 4)} (V)")

        nombre_img = imagen_path / f"Filamento_{i + 1}_roto_reset_{num_simulation}.png"
        Representate.RepresentateState(actual_state, round(voltage, 3), str(nombre_img))

        nombre_pkl = pkl_path / f"filamento_{i + 1}_roto_reset_{num_simulation}.pkl"
        with open(nombre_pkl, "wb") as f:
            pickle.dump(actual_state, f)

    return None


In [ ]:
def generate_oxygen(oxygen_state: np.ndarray, num_oxygen: int):
    eje_y = oxygen_state.shape[1]

    # Generar todas las coordenadas y al mismo tiempo
    y_indices = np.random.randint(0, eje_y, size=num_oxygen)

    # Generar todas las probabilidades de una sola vez
    rand_vals = np.random.rand(num_oxygen)

    # Crear una máscara para las posiciones donde se colocarán oxígenos
    mask = (oxygen_state[y_indices, 0] == 0) & (rand_vals < 0.5)

    # Asignar oxígeno en las posiciones seleccionadas
    oxygen_state[y_indices[mask], 0] = 1

    return oxygen_state


In [ ]:
def update_state_generate(
    state: np.ndarray,
    params: SimulationParameters,
    sim_ctes_cte: dict,
    E_field_vector: np.ndarray,
    temperatura: float,
    factor_vecinos: float,
    factor_sin_vecinos: float,
) -> np.ndarray:
    
    assert E_field_vector.shape[0] == state.shape[0], "El vector de campo eléctrico debe tener igual número de filas que la matriz de estado"
        
    act_state = state.copy()
    
    # Máscara posiciones libres (True donde actual_state == 0)
    free_mask = state == 0
    
    # Máscara vecinos a la izquierda: desplazamos matriz a la derecha
    left_neighbor = np.zeros_like(state, dtype=bool)
    left_neighbor[:, 1:] = state[:, :-1] == 1
    
    # Máscara vecinos a la derecha: desplazamos matriz a la izquierda
    right_neighbor = np.zeros_like(state, dtype=bool)
    right_neighbor[:, :-1] = state[:, 1:] == 1
    
    # Máscara posiciones libres con vecino (True donde hay vecino)
    vecino_mask = left_neighbor | right_neighbor
    
    free_with_vecino = free_mask & vecino_mask
    free_without_vecino = free_mask & (~vecino_mask)

    # calcular las probabilidades por fila
    prob_generacion_fila = np.minimum(
        [
            Generation.Generate(
                params.paso_temporal,
                E_field_vector[i],
                temperatura,
                **sim_ctes_cte,
            )
            for i in range(params.x_size)
        ],
        1,
    )
    
    # Probabilidad base (40x1) expandida a (40x40)
    prob_base_matrix = np.tile(prob_generacion_fila.reshape(-1, 1), (1, params.y_size))
    
    # Crear una copia para la matriz de salida
    prob_final = np.zeros_like(prob_base_matrix)

    # Aplicar factor de vecinos y sin vecinos
    prob_final[free_with_vecino] = prob_base_matrix[free_with_vecino] * factor_vecinos
    prob_final[free_without_vecino] = (prob_base_matrix[free_without_vecino] * factor_sin_vecinos)
    
    # Generar números aleatorios
    aleatorios = np.random.rand(params.x_size, params.y_size)
    
    # Determinar nuevas vacantes
    nueva_vacante = aleatorios < prob_final
    
    # Actualizar el estado
    act_state[nueva_vacante] = 1

    return act_state

In [ ]:
def update_state_recombinate(
    voltage: float,
    E_field: float,
    oxygen_config : dict,
    sim_ctes_dict: dict, 
    params: SimulationParameters, 
    actual_state: np.ndarray,
    oxygen_state: np.ndarray, 
    temperatura: float, 
    ) -> tuple[np.ndarray, np.ndarray]: # type: ignore
    
    """
    Updates the state of the system by generating oxygen, moving oxygen atoms, 
    and recombining states based on the provided parameters.
    Args:
        voltage (float): The applied voltage in the system.
        E_field (float): The electric field in the system.
        oxygen_config (dict): A dictionary mapping threshold voltages to the 
            number of oxygen atoms to generate when the threshold is exceeded.
        sim_ctes_dict (dict): A dictionary containing simulation constants.
        params (SimulationParameters): An object containing simulation parameters 
            such as time step and atom size.
        actual_state (np.ndarray): The current state of the system.
        oxygen_state (np.ndarray): The current state of oxygen atoms in the system.
        temperatura (float): The temperature of the system.
    Returns:
        tuple: A tuple containing:
            - actual_state_update (np.ndarray): The updated state of the system.
            - oxygen_state_update (np.ndarray): The updated state of oxygen atoms.
    """
    
    # Genera oxígenos según voltaje
    for threshold_voltage, num_oxigenos in oxygen_config.items():
        if abs(voltage) > threshold_voltage:
            oxygen_state = generate_oxygen(oxygen_state, num_oxigenos)

        # Muevo los oxígenos
        oxygen_state, velocidad = Recombination.update_oxygen_state(
            paso_temp=params.paso_temporal,
            oxygen_state=oxygen_state,
            temperature=temperatura,
            E_field=E_field,
            grid_size = params.atom_size,
            **sim_ctes_dict,
        )

        # Obtengo la nueva configuración
        actual_state_update, oxygen_state_update = Recombination.Recombine_opt(
            vacancy_state=actual_state,
            oxygen_state=oxygen_state,
            paso_temp=params.paso_temporal,
            velocidad = velocidad,
            temp=temperatura,
            **sim_ctes_dict,
        )
        
        return actual_state_update, oxygen_state_update

In [ ]:
def guardar_datos(
    voltaje_final: float,
    config_state: np.ndarray,
    datos_save: np.ndarray,
    header_files: str,
    save_path_data: Path,
    save_path_pkl: Path,
    save_path_figures: Path,
) -> None:
    
    """
    Saves simulation data, configuration state, and generates a representation of the final state.
    Args:
        voltaje_final_set (float): The final voltage value used in the simulation.
        config_state (np.ndarray): The final configuration state of the simulation.
        datos_save (np.ndarray): The data to be saved as a CSV file.
        header_files (str): The header string for the CSV file.
        save_path_data (Path): The file path where the configuration state will be saved as a binary file.
        save_path_pkl (Path): The file path where the simulation data will be saved as a CSV file.
        save_path_figures (Path): The directory path where the final state representation will be saved.
    Returns:
        None
    """
    
    # Cuando acaba la simulacion guardo el estado final de configuracion
    with open(str(save_path_pkl), "wb") as f:
        pickle.dump(config_state, f)
        
    np.savez(
        save_path_data.with_suffix(".npz"),
        datos=datos_save,
        header=np.array([header_files]),
    )    
    Representate.RepresentateState(config_state, voltaje_final, str(save_path_figures))
    
    return None

In [ ]:
def guardar_representar_estado(
    voltaje: float,
    config_state: np.ndarray,
    save_path_pkl: Path,
    save_path_figures: Path,
) -> None:
    """
    Saves simulation data, configuration state, and generates a representation of the final state.
    Args:
        voltaje_ (float): The voltage value in the state.
        config_state (np.ndarray): The final configuration state of the simulation.
        save_path_pkl (Path): The file path where the simulation data will be saved as a CSV file.
        save_path_figures (Path): The directory path where the final state representation will be saved.
    Returns:
        None
    """

    # Cuando acaba la simulacion guardo el estado final de configuracion
    with open(str(save_path_pkl), "wb") as f:
        pickle.dump(config_state, f)

    Representate.RepresentateState(config_state, voltaje, str(save_path_figures))

    return None

In [ ]:
@medir_tiempo
def PP_set(
    num_simulation: int,
    params: SimulationParameters,
    sim_ctes: SimulationConstants,
    CF_ranges: List[tuple],
    CF_creado: np.ndarray,
):
    """
    Executes the first part (PP) of the simulation set process for a resistive random-access memory (RRAM) device.
    This function simulates the physical processes involved in the formation of conductive filaments (CFs) 
    in an RRAM device during the set operation. It initializes the simulation environment, updates the 
    state of the system, and records the results at each simulation step.
    Args:
        num_simulation (int): The simulation number (used for naming and saving results).
        params (SimulationParameters): Object containing simulation parameters such as device size, 
            total simulation time, number of steps, and voltage settings.
        sim_ctes (SimulationConstants): Object containing simulation constants such as gamma, 
            factor of generation, and material properties.
        CF_ranges (List[tuple]): List of tuples defining the ranges for conductive filaments.
        CF_creado (np.ndarray): Boolean array indicating whether each conductive filament has been created.
    Raises:
        exceptions.MaxVacantesException: Raised if the maximum number of vacancies is exceeded.
        exceptions.NoPercolationException: Raised if the system does not percolate.
        exceptions.HighPercolationVoltageException: Raised if the percolation voltage is too high.
        exceptions.NullResistanceException: Raised if the resistance becomes null during the simulation.
    Notes:
        - The function creates directories for saving simulation results and figures.
        - It uses various helper functions and classes for tasks such as representing the state, 
          solving currents, and calculating probabilities.
        - The simulation stops early if certain conditions are met, such as exceeding the maximum 
          number of vacancies or reaching the final set voltage.
    Returns:
        None
    """

    # Declaro todas las variables que voy a usar exclusivamente en la primera parte (PP) del set.
    rutas = crear_rutas_simulacion(num_simulation = num_simulation, state="set") 

    # Cargo y represento el estado inicial de configuración
    actual_state = cargar_y_representar_estado(
        Path.cwd() / f"Init_data/init_state_{num_simulation-1}",
        rutas["figures_path"] / f"Initial_state_{num_simulation}.png",
        params.initial_voltaje,
    )
    
    sistema_percola = False
    total_vacantes_pp_set = False

    ocupacion_max_pp_set = 0.35
    factor_vecinos = 1.1  # Factor de aumento de la probabilidad si tiene vecino
    factor_libre = 0.9  # Factor de disminución de la probabilidad si no tiene vecino
    lim_voltage_percolacion = 0.75  # Si el voltaje de percolación es mayor que este valor la simulación no vale la pena seguirla
    temperatura = params.T_0
    current = 0.0
    
    max_vancantes_pp_set = int(ocupacion_max_pp_set * params.num_max_vacantes)
    voltage_CF_creado = np.full(len(CF_ranges), 0.0)

    # Inicializo vectores donde almaceno datos
    E_field_vector = np.zeros((actual_state.shape[0]), dtype=np.float64)
    vector_ddp = np.arange(0.000, params.voltaje_final_reset + params.paso_potencial, params.paso_potencial)
    
    num_columnas = 3  # Tiempo, Voltaje, Intensidad
    # Defino la matriz para almacenar los datos
    data_pp_set = np.zeros((params.num_pasos, num_columnas), dtype=np.float64)
    # config_matrix_pp_set = np.zeros((int((params.num_pasos / params.paso_guardar)), params.x_size, params.y_size))
    
    contador_filamentos = 1
    params_dict = asdict(params)
    sim_ctes_dict = asdict(sim_ctes)
         
    print("El nuevo valor de gamma es:", sim_ctes_dict["gamma"], "\n")
                
    indice_gamma = 0

    print(f"\nSimulacion {num_simulation} - Primera parte del set")
    
    for k in range(0, params.num_pasos+1):
        total_vacantes = np.sum(actual_state)
        
        if total_vacantes > int(params.num_max_vacantes):
            # Si se llena el 90 del espacio de la matriz salto a otra simulación. Ponerlo aqui puede dar el problema de que nada mas empezar esté lleno y de error, pero eso NO debe pasar asi q no me preocupa.
            raise exceptions.MaxVacantesException(k=k-1, voltage=vector_ddp[k-1])
        else:
            # Verifica si el sistema ha percolado
            if ((k == params.num_pasos - 1) and (not sistema_percola)):
                raise exceptions.NoPercolationException()

        # Actualizo el tiempo de simulación y el voltaje
        simulation_time = params.paso_temporal * k
        voltage = vector_ddp[k]

        # Genero el vector campo eléctrico
        for i in range(0, params.x_size):
            E_field_vector[i] = ElectricField.GapElectricField(voltage, i, actual_state, **params_dict)
        
        # Verifica si el sistema ha percolado
        if voltage >= params.voltaje_final_set: 
            if not sistema_percola:
                raise exceptions.NoPercolationException()

            voltaje_max_set = vector_ddp[k]
            tiempo_pp_set = params.paso_temporal * (k - 1)  # Le quitamos un paso porque se ha superado el voltaje de ruptura

            print(
                "Voltaje final set",
                voltaje_max_set,
                "en el tiempo",
                tiempo_pp_set,
                "\n",
            )
            # Elimino las filas sobrantes del array de datos y las lleno de nans para eliminarlas luego
            data_pp_set[k - 1 :] = np.nan  # Añadir valores nulos a partir de la fila k
            data_pp_set = data_pp_set[~np.isnan(data_pp_set).any(axis=1)]  # Eliminar filas con valores nulos
            break

        # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
        if Percolation.is_path(actual_state):
            # Si es la primera vez que percola, siste_percola será falso y entra aquí
            if sistema_percola is False:
                voltaje_percolacion = voltage  # Guardo el voltaje de percolación
                print(
                    "\nEl sistema ha percolado en la iteración:",
                    k,
                    " que corresponde con el voltaje:",
                    round(voltaje_percolacion,4),
                    " con una ocupación del:",
                    round((np.sum(actual_state) / (params.num_max_vacantes)), 4) * 100,
                    "%\n",
                )
                
                if voltaje_percolacion >= lim_voltage_percolacion:
                    # Si el voltaje de percolación es demasiado alto no va a coincidir con los datos experimentales, y no merece la pena seguir con la simulación
                    raise exceptions.HighPercolationVoltageException(voltage_percola=voltaje_percolacion)
                
                Representate.RepresentateState(
                    actual_state,
                    round(voltaje_percolacion, 3),
                    str(rutas["figures_path"])
                    + f"/Percola_state_{num_simulation}.png",
                )

                nueva_gamma = sim_ctes.gamma - 1 # / sim_ctes.factor_generacion
                sim_ctes = sim_ctes.update_gamma(nueva_gamma)
                sim_ctes_dict = asdict(sim_ctes)
                
                indice_gamma = indice_gamma + 1
                
                print("El nuevo valor de gamma es:", sim_ctes_dict["gamma"], "\n")

            sistema_percola = True

            _, CF_graph = CurrentSolver.Clean_state_matrix(actual_state)
            filamentos = CurrentSolver.Clasificar_CF(CF_graph, params.x_size, params.y_size, CF_ranges)
            exist_cf = CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges))
            
            # Compruebo si hay filamentos nuevos
            if any(~CF_creado):  
                procesar_filamentos_creados(
                    imagen_path=rutas["figures_path"],
                    pkl_path=rutas["data_simulation_path"],
                    existentes=exist_cf,
                    CF_creado=CF_creado,
                    voltage=voltage,
                    voltage_CF_creado=voltage_CF_creado,
                    actual_state=actual_state,
                    num_simulation=num_simulation)

            
            if sum(CF_creado) == indice_gamma:
                nueva_gamma = sim_ctes.gamma - 1
                sim_ctes = sim_ctes.update_gamma(nueva_gamma)
                sim_ctes_dict = asdict(sim_ctes)
                indice_gamma = indice_gamma + 1
                print("El nuevo valor de gamma es:", sim_ctes_dict["gamma"], "\n")
                
            cf_clean_matrix = CurrentSolver.Eliminar_filamentos_incompletos(CF_graph, CF_ranges, exist_cf)

            # Si ha percolado uso la corriente de Ohm
            try:
                current, _ = CurrentSolver.OmhCurrent(
                    voltage, cf_clean_matrix, **sim_ctes_dict
                )
            except ZeroDivisionError:
                raise exceptions.NullResistanceException(
                    simulation_path=rutas["simulation_path"],
                    figures_path=rutas["figures_path"],
                    voltage=voltage,
                    num_simulation=num_simulation ,
                    actual_state=actual_state,
                )

        else:
            sistema_percola = False
            
            mean_field = np.mean(E_field_vector).item()
            # Si no ha percolado uso la corriente de Poole-Frenkel
            current = CurrentSolver.Poole_Frenkel(temperatura, mean_field ,**sim_ctes_dict
            ) * (params.device_size)

        # Obtengo los valores del campo eléctrico y la temperatura
        temperatura = Temperature.Temperature_Joule(
            voltage, current, sistema_percola, params.T_0, **sim_ctes_dict
        )
        if total_vacantes < max_vancantes_pp_set:
            # Actualizo el estado del sistema
            actual_state = update_state_generate(actual_state, params, sim_ctes_dict, E_field_vector, temperatura, factor_vecinos, factor_libre)
        elif not total_vacantes_pp_set:
            print(
                f"\nSe ha alcanzado la ocupación máxima del {ocupacion_max_pp_set*100}% en la primera parte del set en el paso {k}.\n"
            )
            total_vacantes_pp_set = True

        # Guardo los datos de la simulación
        data_pp_set[k] = np.array([simulation_time, voltage, current])

    # Guardo los datos de la simulación
    save_path_pkl = rutas["data_simulation_path"] / f"Data_pp_set_{num_simulation}.pkl"
    save_path_data = rutas["simulation_path"] / f"Data_pp_set_{num_simulation}.txt"
    save_path_figures = rutas["figures_path"] / f"Final_state_pp_set_{num_simulation}.png"
    
    guardar_datos(
        voltaje_final=params.voltaje_final_set,
        config_state=actual_state,
        datos_save=data_pp_set,
        header_files="Tiempo simulacion [s],Voltaje [V],Intensidad [A]",
        save_path_data=save_path_data,
        save_path_pkl=save_path_pkl,
        save_path_figures=save_path_figures,
    )
    
    # Guardo todas las variables del estado final del PP set para usarlas en el PS set
    final_state_pp_set = {
        "actual_state": actual_state,
        "sistema_percola": sistema_percola,
        "k_maxima": k,
        "sim_ctes": sim_ctes,
        "params": params,
        "Temperatura_final": temperatura,
        "voltaje_max_set": voltaje_max_set,
        "voltaje_percolacion": voltaje_percolacion,
        "tiempo_pp_set": simulation_time,
    }
    with open(rutas["simulation_path"] / f"final_state_pp_set_{num_simulation}.pkl", "wb") as f:
        pickle.dump(actual_state, f)
        
    Representate.RepresentateState(actual_state, round(voltage, 3), str(rutas["figures_path"]) + f"/final_state_pp_set_{num_simulation}.png")
        
    return final_state_pp_set

In [ ]:
@medir_tiempo
def SP_set(
    final_state_pp_set: dict,
    num_simulation: int,
    CF_ranges: List[tuple],
) -> dict:
    """
    Simulates the second part of the "set" process in a resistive switching device.
    This function performs a simulation of the "set" process, updating the state of the system
    based on the applied voltage, electric field, and other parameters. It handles the generation
    of vacancies, checks for percolation, calculates the current, and updates the temperature
    and system state. The results of the simulation are saved to files, and the final state is
    returned for further use.
    Args:
        final_state_pp_set (dict): A dictionary containing the final state of the previous
            simulation step (PP set). It includes the following keys:
            - "actual_state": The current state of the system.
            - "k_maxima": The maximum number of simulation steps.
            - "sistema_percola": Boolean indicating if the system has percolated.
            - "sim_ctes": Simulation constants.
            - "params": Simulation parameters.
            - "voltaje_max_set": Maximum voltage for the set process.
            - "Temperatura_final": Final temperature from the previous step.
        num_simulation (int): The simulation number, used for saving results.
        CF_ranges (List[tuple]): A list of tuples defining the ranges for classifying conductive filaments.
    Returns:
        dict: A dictionary containing the final state of the system after the "set" process.
        It includes the following keys:
            - "actual_state": The updated state of the system.
            - "sim_ctes": Updated simulation constants.
            - "params": Simulation parameters.
            - "Temperatura_final": Final temperature after the "set" process.
    Raises:
        exceptions.MaxVacantesException: If the maximum number of vacancies is exceeded.
        exceptions.NoPercolationException: If the system does not percolate by the end of the simulation.
        exceptions.NullResistanceException: If a division by zero occurs during current calculation.
    Notes:
        - The function saves the simulation data, final state, and figures to specified paths.
        - It uses various helper functions and classes for tasks such as electric field calculation,
          current calculation, and state updates.
        - The simulation stops early if the maximum vacancy occupancy is reached.
    """

    # Extraigo las variables del estado final del PP set
    actual_state = final_state_pp_set["actual_state"]
    k_max = final_state_pp_set["k_maxima"] - 1
    sistema_percola = final_state_pp_set["sistema_percola"]
    sim_ctes = final_state_pp_set["sim_ctes"]
    params = final_state_pp_set["params"]
    voltaje_max_set = final_state_pp_set["voltaje_max_set"]
    tiempo_pp_set = final_state_pp_set["tiempo_pp_set"]

    temperatura = final_state_pp_set["Temperatura_final"]
    
    ocupacion_max_sp_set = 0.4
    max_vancantes_sp_set = int(ocupacion_max_sp_set * params.num_max_vacantes)
    factor_vecinos = 1.2  # Factor de aumento de la probabilidad si tiene vecino
    factor_libre = 0.8  # Factor de disminución de la probabilidad si no tiene vecino
    total_vacantes_sp_set = False
    num_columnas = 3  # Tiempo, Voltaje, Intensidad
    
    rutas = crear_rutas_simulacion(num_simulation = num_simulation, state="set") 
    
    vector_ddp = np.arange(voltaje_max_set, 0.000, -params.paso_potencial)
    E_field_vector = np.zeros((actual_state.shape[0]), dtype=np.float64)
    
    # Defino la matriz para almacenar los datos
    data_sp_set = np.zeros((k_max, num_columnas), dtype=np.float64)
    
    # nueva_gamma = sim_ctes.gamma / sim_ctes.factor_generacion
    # sim_ctes = sim_ctes.update_gamma(nueva_gamma)
    sim_ctes_dict = asdict(sim_ctes)
    
    print(f"Simulacion {num_simulation} - Segunda parte del set\n")    
    for k in range(0, k_max):
        total_vacantes = np.sum(actual_state)
        
        if total_vacantes > int(params.num_max_vacantes):
            # Si se llena el 90 del espacio de la matriz salto a otra simulación. Ponerlo aqui puede dar el problema de que nada mas empezar esté lleno y de error, pero eso NO debe pasar asi q no me preocupa.
            raise exceptions.MaxVacantesException(k=k-1, voltage=vector_ddp[k-1])
        else:
            # Verifica si el sistema ha percolado
            if ((k == params.num_pasos - 1) and (not sistema_percola)):
                raise exceptions.NoPercolationException()

        # Actualizo el tiempo de simulación y el voltaje
        simulation_time = params.paso_temporal * k
        voltage = vector_ddp[k]

        # Genero el vector campo eléctrico
        for i in range(0, params.x_size):
            E_field_vector[i] = ElectricField.GapElectricField(voltage, i, actual_state, **asdict(params))
        
        # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
        if Percolation.is_path(actual_state):           
            _, CF_graph = CurrentSolver.Clean_state_matrix(actual_state)
            filamentos = CurrentSolver.Clasificar_CF(CF_graph, params.x_size, params.y_size, CF_ranges)
            exist_cf = CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges))
            
            cf_clean_matrix = CurrentSolver.Eliminar_filamentos_incompletos(CF_graph, CF_ranges, exist_cf)

            # Si ha percolado uso la corriente de Ohm
            try:
                current, _ = CurrentSolver.OmhCurrent(
                    voltage, cf_clean_matrix, **sim_ctes_dict
                )
            except ZeroDivisionError:
                raise exceptions.NullResistanceException(
                    simulation_path=rutas["simulation_path"],
                    figures_path=rutas["figures_path"],
                    voltage=voltage,
                    num_simulation=num_simulation,
                    actual_state=actual_state,
                )

        else:
            sistema_percola = False
            
            mean_field = np.mean(E_field_vector).item()
            # Si no ha percolado uso la corriente de Poole-Frenkel
            current = CurrentSolver.Poole_Frenkel(temperatura, mean_field ,**sim_ctes_dict
            ) * (params.device_size)

        # Obtengo los valores del campo eléctrico y la temperatura
        temperatura = Temperature.Temperature_Joule(
            voltage, current, sistema_percola, params.T_0, **sim_ctes_dict
        )
        if total_vacantes < max_vancantes_sp_set:
            # Actualizo el estado del sistema
            actual_state = update_state_generate(
                actual_state,
                params,
                sim_ctes_dict,
                E_field_vector,
                temperatura,
                factor_vecinos,
                factor_libre,
            )
        elif not total_vacantes_sp_set:
            print(
                f"Se ha alcanzado la ocupación máxima del {ocupacion_max_sp_set*100}% en la primera segunda del set en el paso {k}.\n"
            )
            total_vacantes_sp_set = True

        # Guardo los datos de la simulación
        data_sp_set[k] = np.array([simulation_time + tiempo_pp_set, voltage, current])
    
    tiempo_sp_set = simulation_time + tiempo_pp_set

    # Guardo los datos de la simulación
    save_path_pkl = rutas["data_simulation_path"] / f"Data_sp_set_{num_simulation}.pkl"
    save_path_data = rutas["simulation_path"] / f"Data_sp_set_{num_simulation}.txt"
    save_path_figures = rutas["figures_path"] / f"Final_state_sp_set_{num_simulation}.png"
    
    guardar_datos(
        voltaje_final=params.voltaje_final_set,
        config_state=actual_state,
        datos_save=data_sp_set,
        header_files="Tiempo simulacion [s],Voltaje [V],Intensidad [A]",
        save_path_data=save_path_data,
        save_path_pkl=save_path_pkl,
        save_path_figures=save_path_figures,
    )
    
    # Guardo todas las variables del estado final del PP set para usarlas en el PS set
    final_state_sp_set = {
        "actual_state": actual_state,
        "sim_ctes": sim_ctes,
        "params": params,
        "Temperatura_final": temperatura,
        "tiempo_sp_set": tiempo_sp_set,
        "percola": sistema_percola,
    }
    with open(rutas["simulation_path"] / f"final_state_sp_set_{num_simulation}.pkl", "wb") as f:
        pickle.dump(actual_state, f)
        
    Representate.RepresentateState(actual_state, round(voltage, 3), str(rutas["figures_path"]) + f"/final_state_sp_set_{num_simulation}.png")
    
    print("\nSimulación del set finalizada correctamente.\n")
        
    return final_state_sp_set

In [ ]:
@medir_tiempo
def PP_reset(
    final_state_sp_set: dict,
    num_simulation: int,
    CF_ranges: List[tuple],
    num_pasos_guardar_estado: int = 3000,
):
    """
    Simulates the reset process of a resistive switching device, updating the system's state and tracking the evolution of various parameters over time.
    Args:
         - final_state_sp_set (dict): A dictionary containing the final state of the set process, including parameters, constants, temperature, time, and other simulation data.
            Expected keys:
                - "params": Simulation parameters.
                - "sim_ctes": Simulation constants.
                - "Temperatura_final": Final temperature from the set process.
                - "tiempo_sp_set": Time elapsed during the set process.
                - "percola": Boolean indicating if percolation occurred.
                - "actual_state": Current state of the system.
                
         - num_simulation (int): The simulation number, used for file naming and tracking.
         - CF_ranges (List[tuple]): A list of tuples defining the ranges for conductive filaments (CFs).
    Returns:
        None. The function performs the simulation, updates the system's state, and saves
        intermediate results (e.g., figures, data files) to disk.
    Notes:
        - During the reset process, no vacancies are generated; only recombination occurs.
        - The simulation involves calculating electric fields, currents, and temperature changes based on the system's state and applied voltage.
        - The state of the system is updated iteratively, and intermediate results are saved every 3000 steps.
        - The function handles both Ohmic and Poole-Frenkel conduction mechanisms depending on whether percolation occurs.
        - If a ZeroDivisionError occurs during current calculation, a custom exception is raised to handle null resistance scenarios.
    """
    
    params = final_state_sp_set["params"]
    sim_ctes = final_state_sp_set["sim_ctes"]
    temperatura = final_state_sp_set["Temperatura_final"]
    tiempo_sp_set = final_state_sp_set["tiempo_sp_set"]
    percola = final_state_sp_set["percola"] # un poco inútil pero bueno, por si acaso (no ha tenido oportunidad de cambir el estado)
    actual_state = final_state_sp_set["actual_state"]
    
    rutas = crear_rutas_simulacion(num_simulation = num_simulation, state="reset")
    
    oxygen_state = np.zeros_like(actual_state, dtype=np.int8)
    
    num_columnas = 3  # Tiempo, Voltaje, Intensidad
    data_pp_reset = np.zeros((params.num_pasos+1, num_columnas), dtype=np.float64)
   
    params_dict = asdict(params)
    sim_ctes_dict = asdict(sim_ctes)

    sim_ctes_dict["voltaje_generar_oxigeno_1"] = 0.7
    sim_ctes_dict["voltaje_generar_oxigeno_2"] = 1.1
    
    # Configuración de umbrales
    oxygen_config = {
        float(sim_ctes_dict["voltaje_generar_oxigeno_1"]): int(sim_ctes_dict["num_oxigenos_pp_reset_1"]),
        float(sim_ctes_dict["voltaje_generar_oxigeno_2"]): int(sim_ctes_dict["num_oxigenos_pp_reset_2"])
    }

    CF_destruido = np.full(len(CF_ranges), False, dtype=bool)
    voltage_CF_destruido = np.full(len(CF_ranges), 0.0)
    
    E_field_vector = np.zeros((actual_state.shape[0]), dtype=np.float64)
    vector_ddp = np.arange(
        -0,
        -(params.voltaje_final_reset + params.paso_potencial),
        -params.paso_potencial,
    )

    print(f"Simulacion {num_simulation} - primera parte del reset")
    
    # Ciclo para la primera parte del reset
    for k in range(0, params.num_pasos+1):
        simulation_time = params.paso_temporal * k
        voltage = vector_ddp[k]

        # Obtengo los valores del campo eléctrico y la temperatura
        E_field = abs(ElectricField.SimpleElectricField(voltage, params.device_size))

        # Genero el vector campo eléctrico
        for i in range(0, actual_state.shape[0]):
            E_field_vector[i] = abs(
                ElectricField.GapElectricField(voltage, i, actual_state, **params_dict)
            )

        _, CF_graph = CurrentSolver.Clean_state_matrix(actual_state)

        max_x, max_y = actual_state.shape
        filamentos = CurrentSolver.Clasificar_CF(CF_graph, max_x, max_y, CF_ranges)
        exist_cf = CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges))

        if any(~CF_destruido):  # mientras haya alguno sin romper
            procesar_filamentos_destruidos(
                    imagen_path=rutas["figures_path"],
                    pkl_path=rutas["data_simulation_path"],
                    existentes=exist_cf,
                    CF_destruido=CF_destruido,
                    voltage=voltage,
                    voltage_CF_destruido=voltage_CF_destruido,
                    actual_state=actual_state,
                    num_simulation=num_simulation
                )

        # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
        if Percolation.is_path(actual_state):
            # Obtengo los caminos de percolación
            cf_clean_matrix = CurrentSolver.Eliminar_filamentos_incompletos(
                CF_graph, CF_ranges, exist_cf)
            percola = True
            
            # Si ha percolado uso la corriente de Ohm
            try:
                current, _ = CurrentSolver.OmhCurrent(
                    voltage, cf_clean_matrix, **sim_ctes_dict
                )
            except ZeroDivisionError:
                raise exceptions.NullResistanceException(
                    simulation_path=rutas["simulation_path"],
                    figures_path=rutas["figures_path"],
                    voltage=voltage,
                    num_simulation=num_simulation,
                    actual_state=actual_state,
                )
        else:
            # Si no ha percolado uso la corriente de Poole-Frenkel
            current = abs(
                CurrentSolver.Poole_Frenkel(
                    temperatura,
                    float(np.mean(E_field_vector)),
                    **sim_ctes_dict,
                ) * (params.device_size)
            )
            
            percola = False
            temperatura = Temperature.Temperature_Joule(voltage, current, percola, params.T_0, **sim_ctes_dict)

        # Actualizo el estado del sistema con la recombinación
        actual_state, oxygen_state = update_state_recombinate(
            voltage = voltage,
            E_field = E_field,
            oxygen_config = oxygen_config,
            sim_ctes_dict = sim_ctes_dict,
            params = params,
            actual_state = actual_state,
            oxygen_state = oxygen_state,
            temperatura = temperatura,
        )
        
        # Tiempo total de la simulacion
        tiempo_total = simulation_time + tiempo_sp_set
        data_pp_reset[k] = np.array([tiempo_total, voltage, current])

        # Represento el estado cada 3000 pasos
        if k % num_pasos_guardar_estado == 0:
            fig_voltage = round(vector_ddp[k], 3)
            guardar_representar_estado(
                voltaje = fig_voltage,
                config_state = actual_state,
                save_path_pkl = rutas["data_simulation_path"] / f"pp_reset_state_V={fig_voltage}_{num_simulation}.pkl",
                save_path_figures = rutas["figures_path"] / f"pp_reset_state_V={fig_voltage}_{num_simulation}.png",
                )
            
            Representate.RepresentateTwoStates(
                matriz1=actual_state, matriz2=oxygen_state, voltage=fig_voltage, filename=str(rutas["figures_path"] / f"pp_reset_full_state_V={fig_voltage}_{num_simulation}.png"))

            print("\nRepresentando el estado de la simulación en el voltaje ", fig_voltage, " (V)" )

    # Guardo los datos de la simulación
    save_path_pkl = rutas["data_simulation_path"] / f"Data_pp_reset_{num_simulation}.pkl"
    save_path_data = rutas["simulation_path"] / f"Data_pp_reset_{num_simulation}.txt"
    save_path_figures = (
        rutas["figures_path"] / f"Final_state_pp_reset_{num_simulation}.png"
    )
    
    guardar_datos(
        voltaje_final=voltage,
        config_state=actual_state,
        datos_save=data_pp_reset,
        header_files="Tiempo simulacion [s],Voltaje [V],Intensidad [A]",
        save_path_data=save_path_data,
        save_path_pkl=save_path_pkl,
        save_path_figures=save_path_figures,
    )

    # Guardo todas las variables del estado final del PP set para usarlas en el PS set
    final_state_pp_reset = {
        "actual_state": actual_state,
        "oxygen_state": oxygen_state,
        "sistema_percola": percola,
        "sim_ctes": sim_ctes,
        "params": params,
        "Temperatura_final": temperatura,
        "voltaje_max_reset": voltage,
        "tiempo_pp_reset": simulation_time,
        "voltage_CF_destruido": voltage_CF_destruido,
    }
    with open(
        rutas["simulation_path"] / f"final_state_pp_reset_{num_simulation}.pkl", "wb"
    ) as f:
        pickle.dump(actual_state, f)

    Representate.RepresentateState(
        actual_state,
        round(voltage, 3),
        str(rutas["figures_path"]) + f"/final_state_pp_reset_{num_simulation}.png",
    )
    
    return final_state_pp_reset


In [ ]:
def SP_reset(
    final_state_pp_reset: dict,
    num_simulation: int,
    CF_ranges: List[tuple],
    num_pasos_guardar_estado: int = 3000
):
    params = final_state_pp_reset["params"]
    sim_ctes = final_state_pp_reset["sim_ctes"]
    temperatura = final_state_pp_reset["Temperatura_final"]
    tiempo_pp_reset = final_state_pp_reset["tiempo_pp_reset"]
    percola = final_state_pp_reset["sistema_percola"]
    actual_state = final_state_pp_reset["actual_state"]
    oxygen_state = final_state_pp_reset["oxygen_state"]
    CF_destruido = final_state_pp_reset["voltage_CF_destruido"]

    rutas = crear_rutas_simulacion(num_simulation=num_simulation, state="reset")

    num_columnas = 3  # Tiempo, Voltaje, Intensidad
    data_pp_reset = np.zeros((params.num_pasos, num_columnas), dtype=np.float64)

    params_dict = asdict(params)
    sim_ctes_dict = asdict(sim_ctes)

    sim_ctes_dict["voltaje_generar_oxigeno"] = -0.2

    # Configuración de umbrales
    oxygen_config = {float(sim_ctes_dict["voltaje_generar_oxigeno"]): int(10)}

    CF_destruido = np.full(len(CF_ranges), False, dtype=bool)
    voltage_CF_destruido = np.full(len(CF_ranges), 0.0)

    E_field_vector = np.zeros((actual_state.shape[0]), dtype=np.float64)
    vector_ddp = np.arange(
        -params.voltaje_final_reset,
        0,
        params.paso_potencial,
    )

    print(f"Simulacion {num_simulation} - segunda parte del reset")

    # Ciclo para la primera parte del reset
    for k in range(0, params.num_pasos):
        simulation_time = params.paso_temporal * k
        voltage = vector_ddp[k]

        # Obtengo los valores del campo eléctrico y la temperatura
        E_field = abs(ElectricField.SimpleElectricField(voltage, params.device_size))

        # Genero el vector campo eléctrico
        for i in range(0, actual_state.shape[0]):
            E_field_vector[i] = abs(
                ElectricField.GapElectricField(voltage, i, actual_state, **params_dict)
            )

        _, CF_graph = CurrentSolver.Clean_state_matrix(actual_state)

        max_x, max_y = actual_state.shape
        filamentos = CurrentSolver.Clasificar_CF(CF_graph, max_x, max_y, CF_ranges)
        exist_cf = CurrentSolver.Existe_filamentos(filamentos, len(CF_ranges))

        if any(~CF_destruido):  # mientras haya alguno sin romper
            procesar_filamentos_destruidos(
                imagen_path=rutas["figures_path"],
                pkl_path=rutas["data_simulation_path"],
                existentes=exist_cf,
                CF_destruido=CF_destruido,
                voltage=voltage,
                voltage_CF_destruido=voltage_CF_destruido,
                actual_state=actual_state,
                num_simulation=num_simulation,
            )

        # Obtengo la corrriente, antes decido cual usar comprobando si ha percolado o no
        if Percolation.is_path(actual_state):
            # Obtengo los caminos de percolación
            cf_clean_matrix = CurrentSolver.Eliminar_filamentos_incompletos(
                CF_graph, CF_ranges, exist_cf
            )
            percola = True

            # Si ha percolado uso la corriente de Ohm
            try:
                current, _ = CurrentSolver.OmhCurrent(voltage, cf_clean_matrix, **sim_ctes_dict)
            except ZeroDivisionError:
                raise exceptions.NullResistanceException(
                    simulation_path=rutas["simulation_path"],
                    figures_path=rutas["figures_path"],
                    voltage=voltage,
                    num_simulation=num_simulation,
                    actual_state=actual_state,
                )
        else:
            # Si no ha percolado uso la corriente de Poole-Frenkel
            current = abs(
                CurrentSolver.Poole_Frenkel(
                    temperatura,
                    float(np.mean(E_field_vector)),
                    **sim_ctes_dict,
                )
                * (params.device_size)
            )

            percola = False
            temperatura = Temperature.Temperature_Joule(
                voltage, current, percola, params.T_0, **sim_ctes_dict
            )

        # Actualizo el estado del sistema con la recombinación
        actual_state, oxygen_state = update_state_recombinate(
            voltage=voltage,
            E_field=E_field,
            oxygen_config=oxygen_config,
            sim_ctes_dict=sim_ctes_dict,
            params=params,
            actual_state=actual_state,
            oxygen_state=oxygen_state,
            temperatura=temperatura,
        )

        # Tiempo total de la simulacion
        tiempo_total = simulation_time + tiempo_pp_reset
        data_pp_reset[k] = np.array([tiempo_total, voltage, current])

        # Represento el estado cada 3000 pasos
        if k % num_pasos_guardar_estado == 0:
            fig_voltage = round(vector_ddp[k], 3)
            guardar_representar_estado(
                voltaje=fig_voltage,
                config_state=actual_state,
                save_path_pkl=rutas["data_simulation_path"]
                / f"sp_reset_state_V={fig_voltage}_{num_simulation}.pkl",
                save_path_figures=rutas["figures_path"]
                / f"sp_reset_state_V={fig_voltage}_{num_simulation}.png",
            )
            
            Representate.RepresentateTwoStates(
                matriz1=actual_state, matriz2=oxygen_state, voltage=fig_voltage, filename=str(rutas["figures_path"] / f"sp_reset_full_state_V={fig_voltage}_{num_simulation}.png"))
            
            print(
                "\nRepresentando el estado de la simulación en el voltaje ",
                fig_voltage,
                " (V)",
            )

    # Guardo los datos de la simulación
    save_path_pkl = (
        rutas["data_simulation_path"] / f"Data_sp_reset_{num_simulation}.pkl"
    )
    save_path_data = rutas["simulation_path"] / f"Data_sp_reset_{num_simulation}.txt"
    save_path_figures = (
        rutas["figures_path"] / f"Final_state_sp_reset_{num_simulation}.png"
    )

    guardar_datos(
        voltaje_final=voltage,
        config_state=actual_state,
        datos_save=data_pp_reset,
        header_files="Tiempo simulacion [s],Voltaje [V],Intensidad [A]",
        save_path_data=save_path_data,
        save_path_pkl=save_path_pkl,
        save_path_figures=save_path_figures,
    )

    with open(
        rutas["simulation_path"] / f"final_state_sp_reset_{num_simulation}.pkl", "wb"
    ) as f:
        pickle.dump(actual_state, f)

    Representate.RepresentateState(
        actual_state,
        round(voltage, 3),
        str(rutas["figures_path"]) + f"/final_state_sp_reset_{num_simulation}.png",
    )
    
    print(f"\nSimulación {num_simulation} finalizada correctamente.\n")

    # Guardo todas las variables del estado final del PP set para usarlas en el PS set
    final_state_sp_reset = {
        "actual_state": actual_state,
        "oxygen_state": oxygen_state,
        "sistema_percola": percola,
        "sim_ctes": sim_ctes,
        "params": params,
        "Temperatura_final": temperatura,
        "tiempo_sp_reset": simulation_time,
        "voltage_CF_destruido": voltage_CF_destruido,
        "CF_destruido": CF_destruido,
    }
    
    return None

In [ ]:
ruta_raiz = Path.cwd()

num_simulation = 0
carpeta_results = ruta_raiz / "Results copy"

    
#Elimino la capreta de resultados anterior
if (carpeta_results).exists():
    shutil.rmtree(carpeta_results)
        

sim_parmtrs = read_csv_to_dic("Init_data/simulation_parameters.csv")
params = SimulationParameters.from_dict(sim_parmtrs[num_simulation])

sim_cte = read_csv_to_dic("Init_data/simulation_constants.csv")
ctes = SimulationConstants.from_dict(sim_cte[num_simulation])
filamentos_ranges = [(0, 9), (10, 19), (20, 29), (30, 39)]
CF_creado = np.full(len(filamentos_ranges), False, dtype=bool)

In [ ]:
final_state_pp_set = PP_set(
    num_simulation=num_simulation + 1,
    params=params,
    sim_ctes=ctes,
    CF_ranges=filamentos_ranges,
    CF_creado=CF_creado,
)

final_state_sp_set = SP_set(
    final_state_pp_set=final_state_pp_set,
    num_simulation=num_simulation + 1,
    CF_ranges=filamentos_ranges,
)

final_state_pp_reset = PP_reset(
    final_state_sp_set=final_state_sp_set,
    num_simulation=num_simulation + 1,
    CF_ranges=filamentos_ranges,
)

final_state_sp_reset = SP_reset(
    final_state_pp_reset=final_state_pp_reset,
    num_simulation=num_simulation + 1,
    CF_ranges=filamentos_ranges,
)

In [ ]:
def simulation_IV(
    num_simulation: int,
    figures_path: Path,
    simulation_path: Path,
    desplazamiento: dict,
    voltaje_percolacion: float,
    voltage_CF_destruido: list,
):   

    # region Representar datos
    save_path = figures_path / f"I-V_{num_simulation}"
    save_path_marcado = figures_path / f"I-V_{num_simulation}_marcado"

    # Definir nombres base y tipos
    prefixes = ["pp", "sp"]
    stages = ["set", "reset"]

    # Diccionario para guardar los datos cargados
    data = {}

    # Cargar archivos de forma automatizada
    for prefix in prefixes:
        for stage in stages:
            name = f"data_{prefix}_{stage}_{num_simulation}.npz"
            key = f"{prefix}_{stage}"
            data[key] = np.load(simulation_path / name)

    # Extraer y concatenar columnas de interés
    i_set = np.concatenate([data["pp_set"]["datos"][:, 2], data["sp_set"]["datos"][:, 2]])
    v_set = np.concatenate([data["pp_set"]["datos"][:, 1], data["sp_set"]["datos"][:, 1]])
    v_reset = np.concatenate([data["pp_reset"]["datos"][:, 1], data["sp_reset"]["datos"][:, 1]])
    i_reset = np.concatenate([data["pp_reset"]["datos"][:, 2], data["sp_reset"]["datos"][:, 2]])


    # Diccionario de puntos que quieres ubicar
    puntos_x_set = {"a": 1e-6, "b": voltaje_percolacion, "c": 1.1}
    puntos_x_pp_reset = {"d": -0.42, "e": voltage_CF_destruido[0], "f": -1.4}
    puntos_x_sp_reset = {"g": -0.01}

    # Obtener puntos en cada curva
    puntos_set = utils.obtener_puntos_en_curva(
        data["pp_set"]["datos"][:, 1], 
        data["pp_set"]["datos"][:, 2], 
        puntos_x_set
    )
    puntos_x_pp_reset = utils.obtener_puntos_en_curva(
        data["pp_reset"]["datos"][:, 1],
        data["pp_reset"]["datos"][:, 2],
        puntos_x_pp_reset,
    )
    puntos_x_sp_reset = utils.obtener_puntos_en_curva(
        data["sp_reset"]["datos"][:, 1],
        data["sp_reset"]["datos"][:, 2],
        puntos_x_sp_reset,
    )

    # Crear un único diccionario combinando ambos
    puntos_totales = {}
    puntos_totales.update(puntos_set)
    puntos_totales.update(puntos_x_pp_reset)
    puntos_totales.update(puntos_x_sp_reset)

    Representate.plot_IV(
        v_set,
        i_set,
        v_reset,
        i_reset,
        num_simulation,
        titulo_figura="",
        figures_path=str(save_path),
    )

    Representate.plot_IV_marcado(
        v_set,
        i_set,
        v_reset,
        i_reset,
        num_simulation,
        puntos_totales,
        desplazamiento,
        titulo_figura="",
        figures_path=str(save_path_marcado),
    )
    
    return None

In [ ]:
# Diccionario de desplazamiento (dx, dy) para cada punto
desplazamiento = {
    "a": (0.025, 1.0),      # derecha, misma altura
    "b": (-0.005, 0.27),    # izquierda, un poco arriba
    "c": (-0.02, 0.35),     # derecha, un poco abajo
    "d": (0.02, 1.0),       # izquierda, misma altura
    "e": (-0.11, 0.66),     # izquierda, misma altura
    "f": (-0.025, 0.25),    # izquierda, un poco abajo
    "g": (-0.12, 0.6),      # derecha, un poco arriba
}

simulation_IV(
    num_simulation=num_simulation+1,
    figures_path=Path.cwd()
    / "Results copy"
    / f"Simulation_{num_simulation}"
    / "Figures",
    simulation_path=Path.cwd() / "Results copy" / f"Simulation_{num_simulation+1}",
    desplazamiento=desplazamiento,
    voltaje_percolacion=final_state_pp_set["voltaje_percolacion"],
    voltage_CF_destruido=final_state_pp_reset["voltage_CF_destruido"],
)